# COMETS 可視化ノート（ログ読むだけ）

このノートは、`comets_run_beginner.py` などで生成された `comets_runs/<run>/` 配下のCSVを読み込み、プロットします。

前提：
- `total_biomass.csv`
- `media.csv`


In [ ]:
from pathlib import Path

RUNS_ROOT = Path("comets_runs")
RUNS_ROOT.exists(), RUNS_ROOT.resolve()

In [ ]:
runs = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir()])
print("runs:", len(runs))
for p in runs[-20:]:
    print(p.name)

## 1) 読み込むrunを選ぶ

`RUN_NAME` を変更して実行。

In [ ]:
RUN_NAME = "user_run"
RUN_DIR = RUNS_ROOT / RUN_NAME
RUN_DIR.resolve()

In [ ]:
import pandas as pd

tb_path = RUN_DIR / "total_biomass.csv"
media_path = RUN_DIR / "media.csv"

if not tb_path.exists():
    raise FileNotFoundError(tb_path)
if not media_path.exists():
    raise FileNotFoundError(media_path)

total_biomass = pd.read_csv(tb_path)
media = pd.read_csv(media_path)

display(total_biomass.head())
display(media.head())

## 2) バイオマス（Total Biomass）

In [ ]:
import matplotlib.pyplot as plt

x_col = "cycle"
y_cols = [c for c in total_biomass.columns if c != x_col]
if not y_cols:
    raise ValueError(f"No biomass columns found in {tb_path}")

ax = total_biomass.plot(x=x_col, y=y_cols)
ax.set_ylabel("Biomass (gr.)")
ax.set_title(RUN_NAME)
plt.show()

## 3) 培地（Media）

物質（metabolite）ごとに濃度推移を描きます。`TARGETS` を変更して実行。

In [ ]:
required = {"metabolite", "cycle", "conc_mmol"}
missing = required - set(media.columns)
if missing:
    raise ValueError(f"media.csv is missing columns: {sorted(missing)}")

TARGETS = ["glc__D_e", "ac_e", "o2_e", "nh4_e", "pi_e"]
m = media.copy()
m = m[m["metabolite"].isin(TARGETS)]

fig, ax = plt.subplots()
for met, df in m.groupby("metabolite"):
    df = df.sort_values("cycle")
    ax.plot(df["cycle"], df["conc_mmol"], label=met)

ax.set_xlabel("cycle")
ax.set_ylabel("Concentration (mmol)")
ax.set_title(RUN_NAME)
ax.legend()
plt.show()

## 4) 便利：利用可能な物質一覧

In [ ]:
mets = sorted(media["metabolite"].unique().tolist())
print("metabolites:", len(mets))
mets[:100]